# ACORN prediction and SHAP attribution

Predicts field-aligned currents for a given time and attributes the result to
its solar wind drivers.

Set `WHEN` and `AREA` below and run all cells.

- `WHEN` — `"realtime"` or a timestamp. The sci model runs when its inputs are
  available for that window, the op model otherwise.
- `AREA` — `"overall"`, `"regions"` (the twelve evaluation sectors), or an
  `(mlat_range, mlt_range)` tuple.

SHAP values are differences from the training climatology, so they read as
what makes this moment unusual relative to the conditions the model learned.

In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve()))
os.environ.setdefault("SUPERMAG_USERID", "acorn_user")

import inference

WHEN    = "realtime"        # "realtime" | "2023-05-06 05:00:00"
AREA    = "overall"         # "overall" | "regions" | ((65, 75), (21, 2))
CHANNEL = 0                 # 0 = mean field, 1 = predicted std
TOP_N   = 4                 # drivers shown in the lag profiles

REALTIME  = str(WHEN).lower() == "realtime"
TIMESTAMP = None if REALTIME else str(WHEN)

## Prediction

In [ ]:
fac, variant = inference.auto_inference(realtime=REALTIME, timestamp=TIMESTAMP)
mean, std, t, _ = fac.predict(timestamp=TIMESTAMP)

m = mean[0] if mean.ndim == 3 else mean
s = std[0]  if std.ndim  == 3 else std

print(f"model {variant}  |  {t}  |  mean {m.min():.2f}..{m.max():.2f}  "
      f"std {s.min():.2f}..{s.max():.2f}")

inference.testing_polar_plot([m, s], t, ["ACORN mean", "ACORN std"])

## Attribution

All areas are attributed from one data fetch and one background set, so their
values are directly comparable.

In [ ]:
if AREA == "regions":
    results = fac.explain_regions(timestamp=TIMESTAMP, channel=CHANNEL)
elif AREA == "overall":
    results = {"Overall": fac.explain(timestamp=TIMESTAMP, channel=CHANNEL)}
else:
    mlat, mlt = AREA
    e = fac.explain(target="custom", timestamp=TIMESTAMP, channel=CHANNEL,
                    mlat_range=mlat, mlt_range=mlt)
    results = {e["label"]: e}

for lab, e in results.items():
    print(f"{lab:<16} prediction {e['prediction']:.4f}   "
          f"climatology {e['base_value']:.4f}   "
          f"departure {e['prediction'] - e['base_value']:+.4f}")

### Area covered

In [ ]:
if len(results) == 1:
    inference.plot_shap_region(next(iter(results.values())), field=m)
else:
    inference.plot_all_shap_regions(field=m)

### Driver importance

Ranked by net contribution. A driver whose influence reverses across the
lookback largely cancels itself, so the signed sum is what matters; `cancel`
below shows how much cancellation each underwent.

In [ ]:
for lab, e in results.items():
    sgn   = e["param_signed"]
    order = np.argsort(np.abs(sgn))[::-1]
    names, vals = np.array(e["params"])[order], sgn[order]

    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.barh(range(len(names)), vals,
            color=["firebrick" if v >= 0 else "steelblue" for v in vals])
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("net SHAP   (red raises |FAC|, blue lowers)")
    ax.set_title(lab)
    span = np.abs(vals).max() or 1.0
    for i, (v, p) in enumerate(zip(vals, e["signed_pct"][order])):
        ax.text(v + 0.02 * span * (1 if v >= 0 else -1), i, f"{p:.0f}%",
                va="center", fontsize=8, ha="left" if v >= 0 else "right")
    ax.set_xlim(-span * 1.3, span * 1.3)
    plt.tight_layout(); plt.show()

### Summary

`cancel` near ±1 means the driver pushed consistently one way across the
lookback; near 0 means it reversed.

In [ ]:
for lab, e in results.items():
    tbl = pd.DataFrame({
        "net SHAP": e["param_signed"],
        "net %":    e["signed_pct"],
        "cancel":   e["cancellation"],
        "peak lag": e["peak_lag"],
    }, index=e["params"])
    if e["physical"] is not None:
        tbl["value now"] = e["physical"][-1]
    tbl = tbl.reindex(tbl["net SHAP"].abs().sort_values(ascending=False).index)
    print(f"\n{lab}")
    display(tbl.round(2))

### SHAP by lag and driver

In [ ]:
if len(results) == 1:
    e = next(iter(results.values()))
    v, lim = e["shap"], np.abs(e["shap"]).max()
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.pcolormesh(np.arange(v.shape[1] + 1), np.arange(v.shape[0] + 1),
                       v, cmap="bwr", vmin=-lim, vmax=lim)
    ax.set_xticks(np.arange(v.shape[1]) + 0.5)
    ax.set_xticklabels(e["params"], rotation=45, ha="right")
    ax.set_ylabel("lag (minutes before prediction)")
    ax.set_title(e["label"])
    fig.colorbar(im, ax=ax).set_label("SHAP value")
    plt.tight_layout(); plt.show()
else:
    inference.plot_region_lag_heatmaps(results)

### Net SHAP across the grid

One panel per driver, shown when several areas are attributed. Colour scales
are per panel, so read within a panel rather than across.

In [ ]:
if len(results) > 1:
    inference.plot_region_shap_polar(results)

### Lag profiles

SHAP in red, the driver's own value in black, for the strongest contributors.

In [ ]:
for lab, e in results.items():
    top = np.argsort(np.abs(e["param_signed"]))[::-1][:TOP_N]
    lag = np.arange(e["shap"].shape[0])[::-1]

    fig, axes = plt.subplots(TOP_N, 1, figsize=(9, 2.1 * TOP_N), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, top):
        ax.plot(lag, e["shap"][:, i], color="crimson", lw=1.5)
        ax.axhline(0, color="k", lw=0.7)
        ax.set_ylabel("SHAP", color="crimson", fontsize=9)
        ax.tick_params(axis="y", labelcolor="crimson", labelsize=8)
        ax.grid(alpha=0.25)
        if e["physical"] is not None:
            ax2 = ax.twinx()
            ax2.plot(lag, e["physical"][:, i], color="k", lw=1.3)
            ax2.set_ylabel(e["params"][i], fontsize=9)
            ax2.tick_params(axis="y", labelsize=8)
    axes[-1].set_xlabel("minutes before prediction")
    axes[-1].invert_xaxis()
    plt.suptitle(lab)
    plt.tight_layout(); plt.show()